# Data Preparation

This notebook cleans, standardises, and aligns the raw macroeconomic
indicators used for UK GDP nowcasting.

The preparation process includes date standardisation, missing-value
inspection, frequency alignment, and construction of the final modelling
dataset.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
gdp = pd.read_csv("../data/raw/gdp_qoq.csv")

gdp.head()

,date,value,label,year,month,quarter,sourceDataset,updateDate
0,1955 Q2,0.1,1955 Q2,1955,NaN,Q2,QNA,2025-09-29T23:00:00.000Z
1,1955 Q3,1.7,1955 Q3,1955,NaN,Q3,QNA,2025-09-29T23:00:00.000Z
2,1955 Q4,-0.6,1955 Q4,1955,NaN,Q4,QNA,2024-09-29T23:00:00.000Z
3,1956 Q1,1.2,1956 Q1,1956,NaN,Q1,QNA,2025-09-29T23:00:00.000Z
4,1956 Q2,-0.1,1956 Q2,1956,NaN,Q2,QNA,2023-09-28T23:00:00.000Z


In [3]:
gdp.tail()

,date,value,label,year,month,quarter,sourceDataset,updateDate
279,2025 Q1,0.6,2025 Q1,2025,NaN,Q1,QNA,2026-06-29T23:00:00.000Z
280,2025 Q2,0.2,2025 Q2,2025,NaN,Q2,QNA,2025-12-22T00:00:00.000Z
281,2025 Q3,0.1,2025 Q3,2025,NaN,Q3,QNA,2025-12-22T00:00:00.000Z
282,2025 Q4,0.1,2025 Q4,2025,NaN,Q4,QNA,2026-03-30T23:00:00.000Z
283,2026 Q1,0.6,2026 Q1,2026,NaN,Q1,QNA,2026-06-29T23:00:00.000Z


In [4]:
print(gdp.shape)
print(gdp.columns.tolist())
print(gdp.dtypes)
print(gdp.isna().sum())

(284, 8)
['date', 'value', 'label', 'year', 'month', 'quarter', 'sourceDataset', 'updateDate']
date                 str
value            float64
label                str
year               int64
month            float64
quarter              str
sourceDataset        str
updateDate           str
dtype: object
date               0
value              0
label              0
year               0
month            284
quarter            0
sourceDataset      0
updateDate         0
dtype: int64


In [6]:
gdp_clean = gdp[["date", "value"]].copy()
gdp_clean.columns = ["quarter", "gdp_qoq"]

gdp_clean.head()

,quarter,gdp_qoq
0,1955 Q2,0.1
1,1955 Q3,1.7
2,1955 Q4,-0.6
3,1956 Q1,1.2
4,1956 Q2,-0.1


In [10]:
gdp_clean["quarter"] = (
    gdp_clean["quarter"]
    .str.replace(" ", "", regex=False)
)

gdp_clean["quarter"] = pd.PeriodIndex(
    gdp_clean["quarter"],
    freq="Q"
)

print(gdp_clean.dtypes)
gdp_clean.head()

quarter    period[Q-DEC]
gdp_qoq          float64
dtype: object


,quarter,gdp_qoq
0,1955Q2,0.1
1,1955Q3,1.7
2,1955Q4,-0.6
3,1956Q1,1.2
4,1956Q2,-0.1


In [11]:
print("Duplicates:", gdp_clean["quarter"].duplicated().sum())
print("Missing GDP:", gdp_clean["gdp_qoq"].isna().sum())
print("Start:", gdp_clean["quarter"].min())
print("End:", gdp_clean["quarter"].max())

Duplicates: 0
Missing GDP: 0
Start: 1955Q2
End: 2026Q1


In [12]:
expected_quarters = pd.period_range(
    start=gdp_clean["quarter"].min(),
    end=gdp_clean["quarter"].max(),
    freq="Q"
)

missing_quarters = expected_quarters.difference(gdp_clean["quarter"])

print("Missing quarters:", len(missing_quarters))
print(missing_quarters)

Missing quarters: 0
PeriodIndex([], dtype='period[Q-DEC]')


In [13]:
gdp_clean.to_csv(
    "../data/processed/gdp_qoq.csv",
    index=False
)

In [15]:
monthly_gdp = pd.read_csv("../data/raw/gdp_monthly.csv")

monthly_gdp.head()

,date,value,label,year,month,quarter,sourceDataset,updateDate
0,1997 FEB,0.4,1997 FEB,1997,February,NaN,MGDP,2025-10-15T23:00:00.000Z
1,1997 MAR,-0.2,1997 MAR,1997,March,NaN,MGDP,2025-10-15T23:00:00.000Z
2,1997 APR,1.2,1997 APR,1997,April,NaN,MGDP,2025-10-15T23:00:00.000Z
3,1997 MAY,-0.8,1997 MAY,1997,May,NaN,MGDP,2025-10-15T23:00:00.000Z
4,1997 JUN,0.8,1997 JUN,1997,June,NaN,MGDP,2025-10-15T23:00:00.000Z


In [16]:
monthly_gdp.tail()

,date,value,label,year,month,quarter,sourceDataset,updateDate
348,2026 FEB,0.5,2026 FEB,2026,February,NaN,MGDP,2026-07-15T23:00:00.000Z
349,2026 MAR,0.3,2026 MAR,2026,March,NaN,MGDP,2026-05-13T23:00:00.000Z
350,2026 APR,-0.1,2026 APR,2026,April,NaN,MGDP,2026-06-11T23:00:00.000Z
351,2026 MAY,0.0,2026 MAY,2026,May,NaN,MGDP,2026-08-12T23:00:00.000Z
352,2026 JUN,0.3,2026 JUN,2026,June,NaN,MGDP,2026-08-12T23:00:00.000Z


In [17]:
print(monthly_gdp.shape)
print(monthly_gdp.columns.tolist())
print(monthly_gdp.dtypes)
print(monthly_gdp.isna().sum())

(353, 8)
['date', 'value', 'label', 'year', 'month', 'quarter', 'sourceDataset', 'updateDate']
date                 str
value            float64
label                str
year               int64
month                str
quarter          float64
sourceDataset        str
updateDate           str
dtype: object
date               0
value              0
label              0
year               0
month              0
quarter          353
sourceDataset      0
updateDate         0
dtype: int64


In [20]:
monthly_gdp_clean = monthly_gdp[["date", "value"]].copy()

monthly_gdp_clean.columns = ["month", "gdp_monthly"]

monthly_gdp_clean["month"] = pd.to_datetime(
    monthly_gdp_clean["month"],
    format="%Y %b"
).dt.to_period("M")

monthly_gdp_clean.head()

,month,gdp_monthly
0,1997-02,0.4
1,1997-03,-0.2
2,1997-04,1.2
3,1997-05,-0.8
4,1997-06,0.8


In [21]:
print("Duplicates:", monthly_gdp_clean["month"].duplicated().sum())
print("Missing values:", monthly_gdp_clean["gdp_monthly"].isna().sum())
print("Start:", monthly_gdp_clean["month"].min())
print("End:", monthly_gdp_clean["month"].max())

Duplicates: 0
Missing values: 0
Start: 1997-02
End: 2026-06


In [25]:
expected_months = pd.period_range(
    start=monthly_gdp_clean["month"].min(),
    end=monthly_gdp_clean["month"].max(),
    freq="M"
)

missing_months = expected_months.difference(monthly_gdp_clean["month"])

print("Missing months:", len(missing_months))
print(missing_months)

Missing months: 0
PeriodIndex([], dtype='period[M]')


In [26]:
monthly_gdp_clean["quarter"] = monthly_gdp_clean["month"].dt.asfreq("Q")

monthly_gdp_clean["month_in_quarter"] = (
    (monthly_gdp_clean["month"].dt.month - 1) % 3 + 1
)

monthly_gdp_clean["month_in_quarter"] = (
    "M" + monthly_gdp_clean["month_in_quarter"].astype(str)
)
monthly_gdp_clean.head(10)

,month,gdp_monthly,quarter,month_in_quarter
0,1997-02,0.4,1997Q1,M2
1,1997-03,-0.2,1997Q1,M3
2,1997-04,1.2,1997Q2,M1
3,1997-05,-0.8,1997Q2,M2
4,1997-06,0.8,1997Q2,M3
5,1997-07,0.4,1997Q3,M1
6,1997-08,0.0,1997Q3,M2
7,1997-09,0.3,1997Q3,M3
8,1997-10,0.7,1997Q4,M1
9,1997-11,0.3,1997Q4,M2


In [40]:
monthly_gdp_quarterly = monthly_gdp_clean.pivot(
    index="quarter",
    columns="month_in_quarter",
    values="gdp_monthly"
)

monthly_gdp_quarterly = monthly_gdp_quarterly.rename(
    columns={
        1: "gdp_monthly_m1",
        2: "gdp_monthly_m2",
        3: "gdp_monthly_m3"
    }
)

monthly_gdp_quarterly = monthly_gdp_quarterly.reset_index()

monthly_gdp_quarterly.head()

month_in_quarter,quarter,gdp_monthly_m1,gdp_monthly_m2,gdp_monthly_m3
0,1997Q1,NaN,0.4,-0.2
1,1997Q2,1.2,-0.8,0.8
2,1997Q3,0.4,0.0,0.3
3,1997Q4,0.7,0.3,1.0
4,1998Q1,-0.2,0.6,-0.3


In [44]:
monthly_gdp_quarterly.columns.name = None
monthly_gdp_quarterly.head()

,quarter,gdp_monthly_m1,gdp_monthly_m2,gdp_monthly_m3
0,1997Q1,NaN,0.4,-0.2
1,1997Q2,1.2,-0.8,0.8
2,1997Q3,0.4,0.0,0.3
3,1997Q4,0.7,0.3,1.0
4,1998Q1,-0.2,0.6,-0.3


In [45]:
print(monthly_gdp_quarterly.dtypes)
print(monthly_gdp_quarterly.isna().sum())
print(monthly_gdp_quarterly.tail())

quarter           period[Q-DEC]
gdp_monthly_m1          float64
gdp_monthly_m2          float64
gdp_monthly_m3          float64
dtype: object
quarter           0
gdp_monthly_m1    1
gdp_monthly_m2    0
gdp_monthly_m3    0
dtype: int64
    quarter  gdp_monthly_m1  gdp_monthly_m2  gdp_monthly_m3
113  2025Q2            -0.1            -0.1             0.4
114  2025Q3            -0.1            -0.3             0.1
115  2025Q4            -0.1             0.1             0.2
116  2026Q1             0.0             0.5             0.3
117  2026Q2            -0.1             0.0             0.3


In [46]:
monthly_gdp_quarterly.to_csv(
    "../data/processed/gdp_monthly.csv",
    index=False
)

In [47]:
services_output = pd.read_csv("../data/raw/services_output.csv")

services_output.head()

,date,value,label,year,month,quarter,sourceDataset,updateDate
0,1997 FEB,0.5,1997 FEB,1997,February,NaN,MGDP,2025-10-15T23:00:00.000Z
1,1997 MAR,0.1,1997 MAR,1997,March,NaN,MGDP,2025-10-15T23:00:00.000Z
2,1997 APR,0.9,1997 APR,1997,April,NaN,MGDP,2025-10-15T23:00:00.000Z
3,1997 MAY,-1.0,1997 MAY,1997,May,NaN,MGDP,2025-10-15T23:00:00.000Z
4,1997 JUN,1.0,1997 JUN,1997,June,NaN,MGDP,2025-10-15T23:00:00.000Z


In [51]:
print(services_output.shape)
print(services_output.columns.tolist())
print(services_output.dtypes)
print(services_output.isna().sum())



(353, 8)
['date', 'value', 'label', 'year', 'month', 'quarter', 'sourceDataset', 'updateDate']
date                 str
value            float64
label                str
year               int64
month                str
quarter          float64
sourceDataset        str
updateDate           str
dtype: object
date               0
value              0
label              0
year               0
month              0
quarter          353
sourceDataset      0
updateDate         0
dtype: int64


In [52]:
print(services_output.tail())

         date  value     label  year     month  quarter sourceDataset  \
348  2026 FEB    0.5  2026 FEB  2026  February      NaN          MGDP   
349  2026 MAR    0.3  2026 MAR  2026     March      NaN          MGDP   
350  2026 APR   -0.1  2026 APR  2026     April      NaN          MGDP   
351  2026 MAY    0.1  2026 MAY  2026       May      NaN          MGDP   
352  2026 JUN    0.4  2026 JUN  2026      June      NaN          MGDP   

                   updateDate  
348  2026-04-15T23:00:00.000Z  
349  2026-05-13T23:00:00.000Z  
350  2026-07-15T23:00:00.000Z  
351  2026-08-12T23:00:00.000Z  
352  2026-08-12T23:00:00.000Z  


In [53]:
services_clean = services_output[["date", "value"]].copy()

services_clean.columns = ["month", "services"]

services_clean["month"] = pd.to_datetime(
    services_clean["month"],
    format="%Y %b"
).dt.to_period("M")

services_clean.head()

,month,services
0,1997-02,0.5
1,1997-03,0.1
2,1997-04,0.9
3,1997-05,-1.0
4,1997-06,1.0


In [54]:
print("Duplicates:", services_clean["month"].duplicated().sum())
print("Missing values:", services_clean["services"].isna().sum())
print("Start:", services_clean["month"].min())
print("End:", services_clean["month"].max())

Duplicates: 0
Missing values: 0
Start: 1997-02
End: 2026-06


In [55]:
expected_months = pd.period_range(
    start=services_clean["month"].min(),
    end=services_clean["month"].max(),
    freq="M"
)

missing_months = expected_months.difference(services_clean["month"])

print("Missing months:", len(missing_months))
print(missing_months)

Missing months: 0
PeriodIndex([], dtype='period[M]')


In [56]:
def monthly_to_quarterly_m123(df, value_col, prefix):
    temp = df.copy()

    temp["quarter"] = temp["month"].dt.asfreq("Q")

    temp["month_in_quarter"] = (
        (temp["month"].dt.month - 1) % 3 + 1
    )

    quarterly = temp.pivot(
        index="quarter",
        columns="month_in_quarter",
        values=value_col
    )

    quarterly = quarterly.rename(
        columns={
            1: f"{prefix}_m1",
            2: f"{prefix}_m2",
            3: f"{prefix}_m3"
        }
    )

    quarterly = quarterly.reset_index()
    quarterly.columns.name = None

    return quarterly

In [57]:
services_quarterly = monthly_to_quarterly_m123(
    services_clean,
    value_col="services",
    prefix="services"
)

services_quarterly.head()

,quarter,services_m1,services_m2,services_m3
0,1997Q1,NaN,0.5,0.1
1,1997Q2,0.9,-1.0,1.0
2,1997Q3,0.1,0.4,0.2
3,1997Q4,0.4,0.2,1.2
4,1998Q1,-0.5,0.5,-0.7


In [58]:
services_quarterly.to_csv(
    "../data/processed/services_output.csv",
    index=False
)

In [59]:
industrial_production = pd.read_csv(
    "../data/raw/industrial_production.csv"
)

industrial_production.head()

,date,value,label,year,month,quarter,sourceDataset,updateDate
0,1997 FEB,0.1,1997 FEB,1997,February,NaN,MGDP,2025-10-15T23:00:00.000Z
1,1997 MAR,-1.5,1997 MAR,1997,March,NaN,MGDP,2025-10-15T23:00:00.000Z
2,1997 APR,2.3,1997 APR,1997,April,NaN,MGDP,2024-10-10T23:00:00.000Z
3,1997 MAY,-0.8,1997 MAY,1997,May,NaN,MGDP,2025-10-15T23:00:00.000Z
4,1997 JUN,0.4,1997 JUN,1997,June,NaN,MGDP,2024-10-10T23:00:00.000Z


In [61]:
print(industrial_production.shape)
print(industrial_production.columns.tolist())
print(industrial_production.dtypes)
print(industrial_production.isna().sum())


(353, 8)
['date', 'value', 'label', 'year', 'month', 'quarter', 'sourceDataset', 'updateDate']
date                 str
value            float64
label                str
year               int64
month                str
quarter          float64
sourceDataset        str
updateDate           str
dtype: object
date               0
value              0
label              0
year               0
month              0
quarter          353
sourceDataset      0
updateDate         0
dtype: int64


In [62]:
industrial_production.tail()

,date,value,label,year,month,quarter,sourceDataset,updateDate
348,2026 FEB,0.3,2026 FEB,2026,February,NaN,MGDP,2026-05-13T23:00:00.000Z
349,2026 MAR,0.0,2026 MAR,2026,March,NaN,MGDP,2026-07-15T23:00:00.000Z
350,2026 APR,0.4,2026 APR,2026,April,NaN,MGDP,2026-08-12T23:00:00.000Z
351,2026 MAY,-0.7,2026 MAY,2026,May,NaN,MGDP,2026-08-12T23:00:00.000Z
352,2026 JUN,-0.2,2026 JUN,2026,June,NaN,MGDP,2026-08-12T23:00:00.000Z


In [63]:
industrial_clean = industrial_production[["date", "value"]].copy()

industrial_clean.columns = ["month", "industrial_production"]

industrial_clean["month"] = pd.to_datetime(
    industrial_clean["month"],
    format="%Y %b"
).dt.to_period("M")

industrial_clean.head()

,month,industrial_production
0,1997-02,0.1
1,1997-03,-1.5
2,1997-04,2.3
3,1997-05,-0.8
4,1997-06,0.4


In [64]:
print("Duplicates:", industrial_clean["month"].duplicated().sum())
print("Missing values:", industrial_clean["industrial_production"].isna().sum())
print("Start:", industrial_clean["month"].min())
print("End:", industrial_clean["month"].max())

Duplicates: 0
Missing values: 0
Start: 1997-02
End: 2026-06


In [65]:
expected_months = pd.period_range(
    start=industrial_clean["month"].min(),
    end=industrial_clean["month"].max(),
    freq="M"
)

missing_months = expected_months.difference(industrial_clean["month"])

print("Missing months:", len(missing_months))
print(missing_months)

Missing months: 0
PeriodIndex([], dtype='period[M]')


In [66]:
industrial_quarterly = monthly_to_quarterly_m123(
    industrial_clean,
    value_col="industrial_production",
    prefix="industrial_production"
)

industrial_quarterly.head()

,quarter,industrial_production_m1,industrial_production_m2,industrial_production_m3
0,1997Q1,NaN,0.1,-1.5
1,1997Q2,2.3,-0.8,0.4
2,1997Q3,1.8,-1.3,0.7
3,1997Q4,1.4,0.2,0.1
4,1998Q1,0.3,0.8,0.9


In [67]:
industrial_quarterly.to_csv(
    "../data/processed/industrial_production.csv",
    index=False
)

In [68]:
construction_output = pd.read_csv(
    "../data/raw/construction_output.csv"
)

construction_output.head()

,date,value,label,year,month,quarter,sourceDataset,updateDate
0,1997 FEB,0.7,1997 FEB,1997,February,NaN,MGDP,2024-10-10T23:00:00.000Z
1,1997 MAR,1.0,1997 MAR,1997,March,NaN,MGDP,2025-10-15T23:00:00.000Z
2,1997 APR,2.0,1997 APR,1997,April,NaN,MGDP,2025-10-15T23:00:00.000Z
3,1997 MAY,0.5,1997 MAY,1997,May,NaN,MGDP,2023-10-11T23:00:00.000Z
4,1997 JUN,-0.4,1997 JUN,1997,June,NaN,MGDP,2025-10-15T23:00:00.000Z


In [69]:
print(construction_output.shape)
print(construction_output.columns.tolist())
print(construction_output.dtypes)
print(construction_output.isna().sum())

(353, 8)
['date', 'value', 'label', 'year', 'month', 'quarter', 'sourceDataset', 'updateDate']
date                 str
value            float64
label                str
year               int64
month                str
quarter          float64
sourceDataset        str
updateDate           str
dtype: object
date               0
value              0
label              0
year               0
month              0
quarter          353
sourceDataset      0
updateDate         0
dtype: int64


In [70]:
construction_output.tail()

,date,value,label,year,month,quarter,sourceDataset,updateDate
348,2026 FEB,0.3,2026 FEB,2026,February,NaN,MGDP,2026-07-15T23:00:00.000Z
349,2026 MAR,1.4,2026 MAR,2026,March,NaN,MGDP,2026-07-15T23:00:00.000Z
350,2026 APR,-0.1,2026 APR,2026,April,NaN,MGDP,2026-07-15T23:00:00.000Z
351,2026 MAY,-0.8,2026 MAY,2026,May,NaN,MGDP,2026-07-15T23:00:00.000Z
352,2026 JUN,-0.1,2026 JUN,2026,June,NaN,MGDP,2026-08-12T23:00:00.000Z


In [71]:
construction_clean = construction_output[["date", "value"]].copy()

construction_clean.columns = ["month", "construction"]

construction_clean["month"] = pd.to_datetime(
    construction_clean["month"],
    format="%Y %b"
).dt.to_period("M")

construction_clean.head()

,month,construction
0,1997-02,0.7
1,1997-03,1.0
2,1997-04,2.0
3,1997-05,0.5
4,1997-06,-0.4


In [72]:
print("Duplicates:", construction_clean["month"].duplicated().sum())
print("Missing values:", construction_clean["construction"].isna().sum())
print("Start:", construction_clean["month"].min())
print("End:", construction_clean["month"].max())

Duplicates: 0
Missing values: 0
Start: 1997-02
End: 2026-06


In [ ]:
expected_months = pd.period_range(
    start=construction_clean["month"].min(),
    end=construction_clean["month"].max(),
    freq="M"
)

missing_months = expected_months.difference(
    construction_clean["month"]
)

print("Missing months:", len(missing_months))
print(missing_months)

Missing months: 0
PeriodIndex([], dtype='period[M]')


In [74]:
construction_quarterly = monthly_to_quarterly_m123(
    construction_clean,
    value_col="construction",
    prefix="construction"
)

construction_quarterly.head()

,quarter,construction_m1,construction_m2,construction_m3
0,1997Q1,NaN,0.7,1.0
1,1997Q2,2.0,0.5,-0.4
2,1997Q3,-0.5,-0.4,0.5
3,1997Q4,1.6,1.5,1.5
4,1998Q1,0.9,0.3,-0.6


In [75]:
construction_quarterly.to_csv(
    "../data/processed/construction_output.csv",
    index=False
)

In [76]:
retail_sales = pd.read_csv(
    "../data/raw/retail_sales.csv"
)

retail_sales.head()

,date,value,label,year,month,quarter,sourceDataset,updateDate
0,1988 FEB,-2.1,1988 FEB,1988,February,NaN,DRSI,2026-03-27T00:00:00.000Z
1,1988 MAR,-0.1,1988 MAR,1988,March,NaN,DRSI,2026-07-23T23:00:00.000Z
2,1988 APR,2.6,1988 APR,1988,April,NaN,DRSI,2026-07-23T23:00:00.000Z
3,1988 MAY,-0.6,1988 MAY,1988,May,NaN,DRSI,2026-07-23T23:00:00.000Z
4,1988 JUN,1.0,1988 JUN,1988,June,NaN,DRSI,2026-07-23T23:00:00.000Z


In [77]:
print(retail_sales.shape)
print(retail_sales.columns.tolist())
print(retail_sales.dtypes)
print(retail_sales.isna().sum())

(461, 8)
['date', 'value', 'label', 'year', 'month', 'quarter', 'sourceDataset', 'updateDate']
date                 str
value            float64
label                str
year               int64
month                str
quarter          float64
sourceDataset        str
updateDate           str
dtype: object
date               0
value              0
label              0
year               0
month              0
quarter          461
sourceDataset      0
updateDate         0
dtype: int64


In [78]:
retail_sales.tail()

,date,value,label,year,month,quarter,sourceDataset,updateDate
456,2026 FEB,-0.9,2026 FEB,2026,February,NaN,DRSI,2026-07-23T23:00:00.000Z
457,2026 MAR,0.1,2026 MAR,2026,March,NaN,DRSI,2026-05-21T23:00:00.000Z
458,2026 APR,0.2,2026 APR,2026,April,NaN,DRSI,2026-07-23T23:00:00.000Z
459,2026 MAY,1.2,2026 MAY,2026,May,NaN,DRSI,2026-06-18T23:00:00.000Z
460,2026 JUN,1.1,2026 JUN,2026,June,NaN,DRSI,2026-07-23T23:00:00.000Z


In [79]:
retail_clean = retail_sales[["date", "value"]].copy()

retail_clean.columns = ["month", "retail_sales"]

retail_clean["month"] = pd.to_datetime(
    retail_clean["month"],
    format="%Y %b"
).dt.to_period("M")

retail_clean.head()

,month,retail_sales
0,1988-02,-2.1
1,1988-03,-0.1
2,1988-04,2.6
3,1988-05,-0.6
4,1988-06,1.0


In [80]:
print("Duplicates:", retail_clean["month"].duplicated().sum())
print("Missing values:", retail_clean["retail_sales"].isna().sum())
print("Start:", retail_clean["month"].min())
print("End:", retail_clean["month"].max())

Duplicates: 0
Missing values: 0
Start: 1988-02
End: 2026-06


In [81]:
expected_months = pd.period_range(
    start=retail_clean["month"].min(),
    end=retail_clean["month"].max(),
    freq="M"
)

missing_months = expected_months.difference(
    retail_clean["month"]
)

print("Missing months:", len(missing_months))
print(missing_months)

Missing months: 0
PeriodIndex([], dtype='period[M]')


In [82]:
retail_quarterly = monthly_to_quarterly_m123(
    retail_clean,
    value_col="retail_sales",
    prefix="retail_sales"
)

retail_quarterly.head()

,quarter,retail_sales_m1,retail_sales_m2,retail_sales_m3
0,1988Q1,NaN,-2.1,-0.1
1,1988Q2,2.6,-0.6,1.0
2,1988Q3,1.5,0.0,-0.7
3,1988Q4,1.6,0.6,0.7
4,1989Q1,-2.1,1.2,-0.1


In [83]:
retail_quarterly.to_csv(
    "../data/processed/retail_sales.csv",
    index=False
)

In [84]:
unemployment_rate = pd.read_csv(
    "../data/raw/unemployment_rate.csv"
)

unemployment_rate.head()

,date,value,label,year,month,quarter,sourceDataset,updateDate
0,1971 FEB,3.8,1971 JAN-MAR,1971,February,NaN,LMS,2015-10-13T23:00:00.000Z
1,1971 MAR,3.9,1971 FEB-APR,1971,March,NaN,LMS,2015-10-13T23:00:00.000Z
2,1971 APR,4.0,1971 MAR-MAY,1971,April,NaN,LMS,2015-10-13T23:00:00.000Z
3,1971 MAY,4.1,1971 APR-JUN,1971,May,NaN,LMS,2015-10-13T23:00:00.000Z
4,1971 JUN,4.1,1971 MAY-JUL,1971,June,NaN,LMS,2015-10-13T23:00:00.000Z


In [85]:
print(unemployment_rate.shape)
print(unemployment_rate.columns.tolist())
print(unemployment_rate.dtypes)
print(unemployment_rate.isna().sum())

(663, 8)
['date', 'value', 'label', 'year', 'month', 'quarter', 'sourceDataset', 'updateDate']
date                 str
value            float64
label                str
year               int64
month                str
quarter          float64
sourceDataset        str
updateDate           str
dtype: object
date               0
value              0
label              0
year               0
month              0
quarter          663
sourceDataset      0
updateDate         0
dtype: int64


In [86]:
unemployment_rate.tail()

,date,value,label,year,month,quarter,sourceDataset,updateDate
658,2025 DEC,5.2,2025 NOV-JAN,2025,December,NaN,LMS,2026-03-19T00:00:00.000Z
659,2026 JAN,4.9,2025 DEC-FEB,2026,January,NaN,LMS,2026-04-20T23:00:00.000Z
660,2026 FEB,5.0,2026 JAN-MAR,2026,February,NaN,LMS,2026-05-18T23:00:00.000Z
661,2026 MAR,4.9,2026 FEB-APR,2026,March,NaN,LMS,2026-06-17T23:00:00.000Z
662,2026 APR,4.9,2026 MAR-MAY,2026,April,NaN,LMS,2026-07-20T23:00:00.000Z


In [87]:
unemployment_clean = unemployment_rate[["date", "value"]].copy()

unemployment_clean.columns = ["month", "unemployment_rate"]

unemployment_clean["month"] = pd.to_datetime(
    unemployment_clean["month"],
    format="%Y %b"
).dt.to_period("M")

unemployment_clean.head()

,month,unemployment_rate
0,1971-02,3.8
1,1971-03,3.9
2,1971-04,4.0
3,1971-05,4.1
4,1971-06,4.1


In [88]:
print("Duplicates:", unemployment_clean["month"].duplicated().sum())
print("Missing values:", unemployment_clean["unemployment_rate"].isna().sum())
print("Start:", unemployment_clean["month"].min())
print("End:", unemployment_clean["month"].max())

Duplicates: 0
Missing values: 0
Start: 1971-02
End: 2026-04


In [89]:
expected_months = pd.period_range(
    start=unemployment_clean["month"].min(),
    end=unemployment_clean["month"].max(),
    freq="M"
)

missing_months = expected_months.difference(
    unemployment_clean["month"]
)

print("Missing months:", len(missing_months))
print(missing_months)

Missing months: 0
PeriodIndex([], dtype='period[M]')


In [90]:
unemployment_quarterly = monthly_to_quarterly_m123(
    unemployment_clean,
    value_col="unemployment_rate",
    prefix="unemployment_rate"
)

unemployment_quarterly.head()

,quarter,unemployment_rate_m1,unemployment_rate_m2,unemployment_rate_m3
0,1971Q1,NaN,3.8,3.9
1,1971Q2,4.0,4.1,4.1
2,1971Q3,4.2,4.2,4.3
3,1971Q4,4.4,4.4,4.5
4,1972Q1,4.5,4.5,4.5


In [91]:
unemployment_quarterly.to_csv(
    "../data/processed/unemployment_rate.csv",
    index=False
)

In [92]:
vacancies = pd.read_csv(
    "../data/raw/vacancies.csv"
)

vacancies.head()

,date,value,label,year,month,quarter,sourceDataset,updateDate
0,2001 MAY,680,2001 APR-JUN,2001,May,NaN,UNEM,2016-03-16T09:30:00.000Z
1,2001 JUN,674,2001 MAY-JUL,2001,June,NaN,UNEM,2023-04-17T23:00:00.000Z
2,2001 JUL,663,2001 JUN-AUG,2001,July,NaN,UNEM,2023-04-17T23:00:00.000Z
3,2001 AUG,664,2001 JUL-SEP,2001,August,NaN,UNEM,2026-04-20T23:00:00.000Z
4,2001 SEP,639,2001 AUG-OCT,2001,September,NaN,UNEM,2022-04-11T23:00:00.000Z


In [93]:
print(vacancies.shape)
print(vacancies.columns.tolist())
print(vacancies.dtypes)
print(vacancies.isna().sum())

(301, 8)
['date', 'value', 'label', 'year', 'month', 'quarter', 'sourceDataset', 'updateDate']
date                 str
value              int64
label                str
year               int64
month                str
quarter          float64
sourceDataset        str
updateDate           str
dtype: object
date               0
value              0
label              0
year               0
month              0
quarter          301
sourceDataset      0
updateDate         0
dtype: int64


In [94]:
vacancies.tail()

,date,value,label,year,month,quarter,sourceDataset,updateDate
296,2026 JAN,725,2025 DEC-FEB,2026,January,NaN,UNEM,2026-06-17T23:00:00.000Z
297,2026 FEB,718,2026 JAN-MAR,2026,February,NaN,UNEM,2026-07-20T23:00:00.000Z
298,2026 MAR,713,2026 FEB-APR,2026,March,NaN,UNEM,2026-07-20T23:00:00.000Z
299,2026 APR,710,2026 MAR-MAY,2026,April,NaN,UNEM,2026-07-20T23:00:00.000Z
300,2026 MAY,712,2026 APR-JUN,2026,May,NaN,UNEM,2026-07-20T23:00:00.000Z


In [95]:
vacancies_clean = vacancies[["date", "value"]].copy()

vacancies_clean.columns = ["month", "vacancies"]

vacancies_clean["month"] = pd.to_datetime(
    vacancies_clean["month"],
    format="%Y %b"
).dt.to_period("M")

vacancies_clean.head()

,month,vacancies
0,2001-05,680
1,2001-06,674
2,2001-07,663
3,2001-08,664
4,2001-09,639


In [96]:
print("Duplicates:", vacancies_clean["month"].duplicated().sum())
print("Missing values:", vacancies_clean["vacancies"].isna().sum())
print("Start:", vacancies_clean["month"].min())
print("End:", vacancies_clean["month"].max())

Duplicates: 0
Missing values: 0
Start: 2001-05
End: 2026-05


In [97]:
expected_months = pd.period_range(
    start=vacancies_clean["month"].min(),
    end=vacancies_clean["month"].max(),
    freq="M"
)

missing_months = expected_months.difference(
    vacancies_clean["month"]
)

print("Missing months:", len(missing_months))
print(missing_months)

Missing months: 0
PeriodIndex([], dtype='period[M]')


In [100]:
vacancies_quarterly = monthly_to_quarterly_m123(
    vacancies_clean,
    value_col="vacancies",
    prefix="vacancies_thousands"
)

vacancies_quarterly.head()

,quarter,vacancies_thousands_m1,vacancies_thousands_m2,vacancies_thousands_m3
0,2001Q2,NaN,680.0,674.0
1,2001Q3,663.0,664.0,639.0
2,2001Q4,626.0,602.0,606.0
3,2002Q1,616.0,619.0,619.0
4,2002Q2,613.0,616.0,611.0


In [101]:
vacancies_quarterly.to_csv(
    "../data/processed/vacancies_quarterly.csv",
    index=False
)

In [102]:
wage_growth = pd.read_csv(
    "../data/raw/wage_growth.csv"
)

wage_growth.head()

,date,value,label,year,month,quarter,sourceDataset,updateDate
0,2001 MAR,6.1,2001 MAR,2001,March,NaN,LMS,2023-09-11T23:00:00.000Z
1,2001 APR,6.2,2001 APR,2001,April,NaN,LMS,2023-09-11T23:00:00.000Z
2,2001 MAY,5.1,2001 MAY,2001,May,NaN,LMS,2017-06-13T23:00:00.000Z
3,2001 JUN,5.5,2001 JUN,2001,June,NaN,LMS,2015-10-13T23:00:00.000Z
4,2001 JUL,5.2,2001 JUL,2001,July,NaN,LMS,2015-10-13T23:00:00.000Z


In [103]:
print(wage_growth.shape)
print(wage_growth.columns.tolist())
print(wage_growth.dtypes)
print(wage_growth.isna().sum())

(303, 8)
['date', 'value', 'label', 'year', 'month', 'quarter', 'sourceDataset', 'updateDate']
date                 str
value            float64
label                str
year               int64
month                str
quarter          float64
sourceDataset        str
updateDate           str
dtype: object
date               0
value              0
label              0
year               0
month              0
quarter          303
sourceDataset      0
updateDate         0
dtype: int64


In [104]:
wage_growth.tail()

,date,value,label,year,month,quarter,sourceDataset,updateDate
298,2026 JAN,4.1,2026 JAN,2026,January,NaN,LMS,2026-04-20T23:00:00.000Z
299,2026 FEB,3.9,2026 FEB,2026,February,NaN,LMS,2026-05-18T23:00:00.000Z
300,2026 MAR,4.4,2026 MAR,2026,March,NaN,LMS,2026-06-17T23:00:00.000Z
301,2026 APR,4.4,2026 APR,2026,April,NaN,LMS,2026-06-17T23:00:00.000Z
302,2026 MAY,4.3,2026 MAY,2026,May,NaN,LMS,2026-07-20T23:00:00.000Z


In [105]:
wage_clean = wage_growth[["date", "value"]].copy()

wage_clean.columns = ["month", "wage_growth"]

wage_clean["month"] = pd.to_datetime(
    wage_clean["month"],
    format="%Y %b"
).dt.to_period("M")

wage_clean.head()

,month,wage_growth
0,2001-03,6.1
1,2001-04,6.2
2,2001-05,5.1
3,2001-06,5.5
4,2001-07,5.2


In [106]:
print("Duplicates:", wage_clean["month"].duplicated().sum())
print("Missing values:", wage_clean["wage_growth"].isna().sum())
print("Start:", wage_clean["month"].min())
print("End:", wage_clean["month"].max())

Duplicates: 0
Missing values: 0
Start: 2001-03
End: 2026-05


In [107]:
expected_months = pd.period_range(
    start=wage_clean["month"].min(),
    end=wage_clean["month"].max(),
    freq="M"
)

missing_months = expected_months.difference(
    wage_clean["month"]
)

print("Missing months:", len(missing_months))
print(missing_months)

Missing months: 0
PeriodIndex([], dtype='period[M]')


In [108]:
wage_quarterly = monthly_to_quarterly_m123(
    wage_clean,
    value_col="wage_growth",
    prefix="wage_growth"
)

wage_quarterly.head()

,quarter,wage_growth_m1,wage_growth_m2,wage_growth_m3
0,2001Q1,NaN,NaN,6.1
1,2001Q2,6.2,5.1,5.5
2,2001Q3,5.2,5.3,4.8
3,2001Q4,4.8,4.6,4.2
4,2002Q1,3.8,3.2,3.0


In [109]:
wage_quarterly.to_csv(
    "../data/processed/wage_growth_quarterly.csv",
    index=False
)

In [110]:
cpi_inflation = pd.read_csv(
    "../data/raw/cpi_inflation.csv"
)

cpi_inflation.head()

,date,value,label,year,month,quarter,sourceDataset,updateDate
0,1989 JAN,4.9,1989 JAN,1989,January,NaN,MM23,2015-10-12T23:00:00.000Z
1,1989 FEB,5.0,1989 FEB,1989,February,NaN,MM23,2015-10-12T23:00:00.000Z
2,1989 MAR,5.0,1989 MAR,1989,March,NaN,MM23,2015-10-12T23:00:00.000Z
3,1989 APR,5.3,1989 APR,1989,April,NaN,MM23,2015-10-12T23:00:00.000Z
4,1989 MAY,5.3,1989 MAY,1989,May,NaN,MM23,2015-10-12T23:00:00.000Z


In [111]:
print(cpi_inflation.shape)
print(cpi_inflation.columns.tolist())
print(cpi_inflation.dtypes)
print(cpi_inflation.isna().sum())

(450, 8)
['date', 'value', 'label', 'year', 'month', 'quarter', 'sourceDataset', 'updateDate']
date                 str
value            float64
label                str
year               int64
month                str
quarter          float64
sourceDataset        str
updateDate           str
dtype: object
date               0
value              0
label              0
year               0
month              0
quarter          450
sourceDataset      0
updateDate         0
dtype: int64


In [112]:
cpi_inflation.tail()

,date,value,label,year,month,quarter,sourceDataset,updateDate
445,2026 FEB,3.0,2026 FEB,2026,February,NaN,MM23,2026-03-20T00:00:00.000Z
446,2026 MAR,3.3,2026 MAR,2026,March,NaN,MM23,2026-04-21T23:00:00.000Z
447,2026 APR,2.8,2026 APR,2026,April,NaN,MM23,2026-05-19T23:00:00.000Z
448,2026 MAY,2.8,2026 MAY,2026,May,NaN,MM23,2026-06-16T23:00:00.000Z
449,2026 JUN,2.6,2026 JUN,2026,June,NaN,MM23,2026-07-21T23:00:00.000Z


In [113]:
cpi_clean = cpi_inflation[["date", "value"]].copy()

cpi_clean.columns = ["month", "cpi_inflation"]

cpi_clean["month"] = pd.to_datetime(
    cpi_clean["month"],
    format="%Y %b"
).dt.to_period("M")

cpi_clean.head()

,month,cpi_inflation
0,1989-01,4.9
1,1989-02,5.0
2,1989-03,5.0
3,1989-04,5.3
4,1989-05,5.3


In [114]:
print("Duplicates:", cpi_clean["month"].duplicated().sum())
print("Missing values:", cpi_clean["cpi_inflation"].isna().sum())
print("Start:", cpi_clean["month"].min())
print("End:", cpi_clean["month"].max())

Duplicates: 0
Missing values: 0
Start: 1989-01
End: 2026-06


In [115]:
expected_months = pd.period_range(
    start=cpi_clean["month"].min(),
    end=cpi_clean["month"].max(),
    freq="M"
)

missing_months = expected_months.difference(
    cpi_clean["month"]
)

print("Missing months:", len(missing_months))
print(missing_months)

Missing months: 0
PeriodIndex([], dtype='period[M]')


In [116]:
cpi_quarterly = monthly_to_quarterly_m123(
    cpi_clean,
    value_col="cpi_inflation",
    prefix="cpi_inflation"
)

cpi_quarterly.head()

,quarter,cpi_inflation_m1,cpi_inflation_m2,cpi_inflation_m3
0,1989Q1,4.9,5.0,5.0
1,1989Q2,5.3,5.3,5.2
2,1989Q3,5.2,5.0,5.2
3,1989Q4,5.5,5.5,5.5
4,1990Q1,5.7,5.9,6.0


In [117]:
cpi_quarterly.to_csv(
    "../data/processed/cpi_inflation_quarterly.csv",
    index=False
)

In [118]:
bank_rate = pd.read_csv(
    "../data/raw/bank_rate.csv"
)

bank_rate.head()

,Date,Official Bank Rate \t\t\t\t\t\t[a] [b] \t\t\t\t\t\tIUDBEDR
0,13-Aug-26,3.75
1,12-Aug-26,3.75
2,11-Aug-26,3.75
3,10-Aug-26,3.75
4,07-Aug-26,3.75


In [119]:
print(bank_rate.shape)
print(bank_rate.columns.tolist())
print(bank_rate.dtypes)
print(bank_rate.isna().sum())

(6725, 2)
['Date', 'Official Bank Rate  \t\t\t\t\t\t[a] [b] \t\t\t\t\t\tIUDBEDR']
Date                                                               str
Official Bank Rate  \t\t\t\t\t\t[a] [b] \t\t\t\t\t\tIUDBEDR    float64
dtype: object
Date                                                           0
Official Bank Rate  \t\t\t\t\t\t[a] [b] \t\t\t\t\t\tIUDBEDR    0
dtype: int64


In [120]:
bank_rate.tail()

,Date,Official Bank Rate \t\t\t\t\t\t[a] [b] \t\t\t\t\t\tIUDBEDR
6720,10-Jan-00,5.5
6721,07-Jan-00,5.5
6722,06-Jan-00,5.5
6723,05-Jan-00,5.5
6724,04-Jan-00,5.5


In [121]:
bank_rate_clean = bank_rate.iloc[:, :2].copy()

bank_rate_clean.columns = ["date", "bank_rate"]

bank_rate_clean.head()

,date,bank_rate
0,13-Aug-26,3.75
1,12-Aug-26,3.75
2,11-Aug-26,3.75
3,10-Aug-26,3.75
4,07-Aug-26,3.75


In [122]:
bank_rate_clean["date"] = pd.to_datetime(
    bank_rate_clean["date"],
    format="%d-%b-%y"
)

bank_rate_clean.head()

,date,bank_rate
0,2026-08-13,3.75
1,2026-08-12,3.75
2,2026-08-11,3.75
3,2026-08-10,3.75
4,2026-08-07,3.75


In [124]:
print(bank_rate_clean.dtypes)

date         datetime64[us]
bank_rate           float64
dtype: object


In [125]:
bank_rate_clean = bank_rate_clean.sort_values("date").reset_index(drop=True)

bank_rate_clean.head()

,date,bank_rate
0,2000-01-04,5.5
1,2000-01-05,5.5
2,2000-01-06,5.5
3,2000-01-07,5.5
4,2000-01-10,5.5


In [126]:
bank_rate_clean.tail()

,date,bank_rate
6720,2026-08-07,3.75
6721,2026-08-10,3.75
6722,2026-08-11,3.75
6723,2026-08-12,3.75
6724,2026-08-13,3.75


In [127]:
bank_rate_clean["month"] = bank_rate_clean["date"].dt.to_period("M")

bank_rate_clean.head()

,date,bank_rate,month
0,2000-01-04,5.5,2000-01
1,2000-01-05,5.5,2000-01
2,2000-01-06,5.5,2000-01
3,2000-01-07,5.5,2000-01
4,2000-01-10,5.5,2000-01


In [128]:
bank_rate_monthly = (
    bank_rate_clean
    .groupby("month", as_index=False)
    .last()
)

bank_rate_monthly.head()

,month,date,bank_rate
0,2000-01,2000-01-31,5.75
1,2000-02,2000-02-29,6.00
2,2000-03,2000-03-31,6.00
3,2000-04,2000-04-28,6.00
4,2000-05,2000-05-31,6.00


In [129]:
expected_months = pd.period_range(
    start=bank_rate_monthly["month"].min(),
    end=bank_rate_monthly["month"].max(),
    freq="M"
)

missing_months = expected_months.difference(
    bank_rate_monthly["month"]
)

print("Missing months:", len(missing_months))
print(missing_months)

Missing months: 0
PeriodIndex([], dtype='period[M]')


In [130]:
bank_rate_monthly = bank_rate_monthly[
    ["month", "bank_rate"]
].copy()

bank_rate_monthly.head()

,month,bank_rate
0,2000-01,5.75
1,2000-02,6.00
2,2000-03,6.00
3,2000-04,6.00
4,2000-05,6.00


In [131]:
bank_rate_quarterly = monthly_to_quarterly_m123(
    bank_rate_monthly,
    value_col="bank_rate",
    prefix="bank_rate"
)

bank_rate_quarterly.head()

,quarter,bank_rate_m1,bank_rate_m2,bank_rate_m3
0,2000Q1,5.75,6.00,6.00
1,2000Q2,6.00,6.00,6.00
2,2000Q3,6.00,6.00,6.00
3,2000Q4,6.00,6.00,6.00
4,2001Q1,6.00,5.75,5.75


In [132]:
print(bank_rate_quarterly.isna().sum())
print(bank_rate_quarterly.tail())

quarter         0
bank_rate_m1    0
bank_rate_m2    0
bank_rate_m3    1
dtype: int64
    quarter  bank_rate_m1  bank_rate_m2  bank_rate_m3
102  2025Q3          4.25          4.00          4.00
103  2025Q4          4.00          4.00          3.75
104  2026Q1          3.75          3.75          3.75
105  2026Q2          3.75          3.75          3.75
106  2026Q3          3.75          3.75           NaN


In [133]:
bank_rate_quarterly.to_csv(
    "../data/processed/bank_rate_quarterly.csv",
    index=False
)

In [134]:
consumer_confidence = pd.read_csv(
    "../data/raw/consumer_confidence.csv"
)

consumer_confidence.head()

,Date,Actual
0,Jul-26,-17
1,Jun-26,-23
2,May-26,-23
3,Apr-26,-25
4,Mar-26,-21


In [135]:
print(consumer_confidence.shape)
print(consumer_confidence.columns.tolist())
print(consumer_confidence.dtypes)
print(consumer_confidence.isna().sum())


(247, 2)
['Date', 'Actual']
Date        str
Actual    int64
dtype: object
Date      0
Actual    0
dtype: int64


In [136]:
consumer_confidence.tail()

,Date,Actual
242,May-06,-5
243,Apr-06,-4
244,Mar-06,-7
245,Feb-06,-4
246,Jan-06,-3


In [137]:
consumer_confidence_clean = consumer_confidence[
    ["Date", "Actual"]
].copy()

consumer_confidence_clean.columns = [
    "month", "consumer_confidence"
]

consumer_confidence_clean.head()

,month,consumer_confidence
0,Jul-26,-17
1,Jun-26,-23
2,May-26,-23
3,Apr-26,-25
4,Mar-26,-21


In [138]:
consumer_confidence_clean["month"] = pd.to_datetime(
    consumer_confidence_clean["month"],
    format="%b-%y"
).dt.to_period("M")

In [139]:
consumer_confidence_clean = (
    consumer_confidence_clean
    .sort_values("month")
    .reset_index(drop=True)
)

consumer_confidence_clean.head()

,month,consumer_confidence
0,2006-01,-3
1,2006-02,-4
2,2006-03,-7
3,2006-04,-4
4,2006-05,-5


In [140]:
print("Duplicates:", consumer_confidence_clean["month"].duplicated().sum())
print("Missing values:", consumer_confidence_clean["consumer_confidence"].isna().sum())
print("Start:", consumer_confidence_clean["month"].min())
print("End:", consumer_confidence_clean["month"].max())

Duplicates: 0
Missing values: 0
Start: 2006-01
End: 2026-07


In [141]:
expected_months = pd.period_range(
    start=consumer_confidence_clean["month"].min(),
    end=consumer_confidence_clean["month"].max(),
    freq="M"
)

missing_months = expected_months.difference(
    consumer_confidence_clean["month"]
)

print("Missing months:", len(missing_months))
print(missing_months)

Missing months: 0
PeriodIndex([], dtype='period[M]')


In [142]:
consumer_confidence_quarterly = monthly_to_quarterly_m123(
    consumer_confidence_clean,
    value_col="consumer_confidence",
    prefix="consumer_confidence"
)

consumer_confidence_quarterly.head()

,quarter,consumer_confidence_m1,consumer_confidence_m2,consumer_confidence_m3
0,2006Q1,-3.0,-4.0,-7.0
1,2006Q2,-4.0,-5.0,-4.0
2,2006Q3,-4.0,-8.0,-7.0
3,2006Q4,-5.0,-7.0,-8.0
4,2007Q1,-7.0,-8.0,-8.0


In [143]:
consumer_confidence_quarterly.to_csv(
    "../data/processed/consumer_confidence_quarterly.csv",
    index=False
)

In [144]:
oecd_cli = pd.read_csv(
    "../data/raw/oecd_cli.csv"
)

oecd_cli.head()

,STRUCTURE,STRUCTURE_ID,STRUCTURE_NAME,ACTION,REF_AREA,Reference area,FREQ,Frequency of observation,MEASURE,Measure,...,OBS_VALUE,Observation value,OBS_STATUS,Observation status,UNIT_MULT,Unit multiplier,DECIMALS,Decimals,BASE_PER,Base period
0,DATAFLOW,OECD.SDD.STES:DSD_STES@DF_CLI(4.1),Composite leading indicators,I,GBR,United Kingdom,M,Monthly,LI,Composite leading indicator (CLI),...,100.8071,NaN,A,Normal value,0,Units,2,Two,NaN,NaN
1,DATAFLOW,OECD.SDD.STES:DSD_STES@DF_CLI(4.1),Composite leading indicators,I,GBR,United Kingdom,M,Monthly,LI,Composite leading indicator (CLI),...,100.8000,NaN,A,Normal value,0,Units,2,Two,NaN,NaN
2,DATAFLOW,OECD.SDD.STES:DSD_STES@DF_CLI(4.1),Composite leading indicators,I,GBR,United Kingdom,M,Monthly,LI,Composite leading indicator (CLI),...,100.8529,NaN,A,Normal value,0,Units,2,Two,NaN,NaN
3,DATAFLOW,OECD.SDD.STES:DSD_STES@DF_CLI(4.1),Composite leading indicators,I,GBR,United Kingdom,M,Monthly,LI,Composite leading indicator (CLI),...,99.3928,NaN,A,Normal value,0,Units,2,Two,NaN,NaN
4,DATAFLOW,OECD.SDD.STES:DSD_STES@DF_CLI(4.1),Composite leading indicators,I,GBR,United Kingdom,M,Monthly,LI,Composite leading indicator (CLI),...,100.6103,NaN,A,Normal value,0,Units,2,Two,NaN,NaN


In [145]:
print(oecd_cli.shape)
print(oecd_cli.columns.tolist())
print(oecd_cli.dtypes)
print(oecd_cli.isna().sum())

(246, 34)
['STRUCTURE', 'STRUCTURE_ID', 'STRUCTURE_NAME', 'ACTION', 'REF_AREA', 'Reference area', 'FREQ', 'Frequency of observation', 'MEASURE', 'Measure', 'UNIT_MEASURE', 'Unit of measure', 'ACTIVITY', 'Economic activity', 'ADJUSTMENT', 'Adjustment', 'TRANSFORMATION', 'Transformation', 'TIME_HORIZ', 'Time horizon', 'METHODOLOGY', 'Calculation methodology', 'TIME_PERIOD', 'Time period', 'OBS_VALUE', 'Observation value', 'OBS_STATUS', 'Observation status', 'UNIT_MULT', 'Unit multiplier', 'DECIMALS', 'Decimals', 'BASE_PER', 'Base period']
STRUCTURE                       str
STRUCTURE_ID                    str
STRUCTURE_NAME                  str
ACTION                          str
REF_AREA                        str
Reference area                  str
FREQ                            str
Frequency of observation        str
MEASURE                         str
Measure                         str
UNIT_MEASURE                    str
Unit of measure                 str
ACTIVITY                 

In [146]:
oecd_cli_clean = oecd_cli[
    ["TIME_PERIOD", "OBS_VALUE"]
].copy()

oecd_cli_clean.head()

,TIME_PERIOD,OBS_VALUE
0,2010-11,100.8071
1,2010-12,100.8000
2,2010-09,100.8529
3,2021-02,99.3928
4,2021-03,100.6103


In [147]:
oecd_cli_clean.columns = ["month", "oecd_cli"]

oecd_cli_clean.head()

,month,oecd_cli
0,2010-11,100.8071
1,2010-12,100.8000
2,2010-09,100.8529
3,2021-02,99.3928
4,2021-03,100.6103


In [148]:
oecd_cli_clean["month"] = pd.to_datetime(
    oecd_cli_clean["month"],
    format="%Y-%m"
).dt.to_period("M")

In [149]:
oecd_cli_clean = (
    oecd_cli_clean
    .sort_values("month")
    .reset_index(drop=True)
)

oecd_cli_clean.head()

,month,oecd_cli
0,2006-01,100.5471
1,2006-02,100.7167
2,2006-03,100.8848
3,2006-04,101.0458
4,2006-05,101.1928


In [150]:
oecd_cli_clean.tail()

,month,oecd_cli
241,2026-02,101.0030
242,2026-03,100.8417
243,2026-04,100.6594
244,2026-05,100.4770
245,2026-06,100.2865


In [151]:
print("Shape:", oecd_cli_clean.shape)
print("Duplicates:", oecd_cli_clean["month"].duplicated().sum())
print("Missing values:", oecd_cli_clean["oecd_cli"].isna().sum())
print("Start:", oecd_cli_clean["month"].min())
print("End:", oecd_cli_clean["month"].max())

Shape: (246, 2)
Duplicates: 0
Missing values: 0
Start: 2006-01
End: 2026-06


In [152]:
expected_months = pd.period_range(
    start=oecd_cli_clean["month"].min(),
    end=oecd_cli_clean["month"].max(),
    freq="M"
)

missing_months = expected_months.difference(
    oecd_cli_clean["month"]
)

print("Expected months:", len(expected_months))
print("Actual rows:", len(oecd_cli_clean))
print("Missing months:", len(missing_months))
print(missing_months)

Expected months: 246
Actual rows: 246
Missing months: 0
PeriodIndex([], dtype='period[M]')


In [153]:
extra_months = oecd_cli_clean["month"][
    ~oecd_cli_clean["month"].isin(expected_months)
]

print("Extra months:", len(extra_months))
print(extra_months)

Extra months: 0
Series([], Name: month, dtype: period[M])


In [154]:
print(oecd_cli_clean.dtypes)

print("Non-finite values:",
      (~np.isfinite(oecd_cli_clean["oecd_cli"])).sum())

print(oecd_cli_clean["oecd_cli"].describe())

month       period[M]
oecd_cli      float64
dtype: object
Non-finite values: 0
count    246.000000
mean      99.812096
std        2.247768
min       86.958440
25%       98.599285
50%      100.421400
75%      101.229375
max      103.299600
Name: oecd_cli, dtype: float64


In [155]:
oecd_cli_quarterly = monthly_to_quarterly_m123(
    oecd_cli_clean,
    value_col="oecd_cli",
    prefix="oecd_cli"
)

oecd_cli_quarterly.head()

,quarter,oecd_cli_m1,oecd_cli_m2,oecd_cli_m3
0,2006Q1,100.5471,100.7167,100.8848
1,2006Q2,101.0458,101.1928,101.3334
2,2006Q3,101.4845,101.6628,101.8751
3,2006Q4,102.1039,102.3324,102.5540
4,2007Q1,102.7589,102.9538,103.1211


In [156]:
oecd_cli_quarterly.tail()

,quarter,oecd_cli_m1,oecd_cli_m2,oecd_cli_m3
77,2025Q2,100.5511,100.7680,100.9816
78,2025Q3,101.1447,101.2447,101.2785
79,2025Q4,101.2679,101.2325,101.1902
80,2026Q1,101.1187,101.0030,100.8417
81,2026Q2,100.6594,100.4770,100.2865


In [157]:
oecd_cli_quarterly.to_csv(
    "../data/processed/oecd_cli_quarterly.csv",
    index=False
)